# Gallstone Disease — Predictive Models Analysis
**Course:** Modelos Predictivos — Parcial I  
**Dataset:** Gallstone UCI (`dataset-uci.csv`)  
**Task:** Binary Classification — predict `Gallstone Status` (0 / 1)  
**Author:** Sergio Florez  
---

## Notebook Index
1. Dataset Description  
   1.1 Context and Objective  
   1.2 Feature Descriptions  
2. Data Loading & Global Cleaning  
   2.1 Load Data  
   2.2 Quality Check  
   2.3 Cleaning Steps  
3. Train/Test Split  
   3.1 Split Configuration  
   3.2 Split Verification  
4. Exploratory Data Analysis (Training Set Only)  
   4.1 Do lab values differ between classes?  
   4.2 Is there dangerous multicollinearity among body composition features?  
   4.3 What is the demographic profile by disease status?  
5. Feature Engineering & Selection  
   5.1 Drop Redundant Features  
   5.2 Correlation-Based Filtering  
   5.3 SelectKBest Ranking  
   5.4 Final Feature Set  
6. Model Training & Cross-Validation  
   6.1 Baseline — DummyClassifier  
   6.2 Model 1 — Logistic Regression  
   6.3 Model 2 — Random Forest  
   6.4 CV Comparison Table  
7. Final Evaluation on Test Set  
   7.1 Metrics Table  
   7.2 Diagnostic Plots  
   7.3 CV vs. Test Comparison  
8. Conclusions

In [ ]:
# Only installs what Colab doesn't have by default
!pip install -q imbalanced-learn xgboost

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings("ignore")

# ── Data manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Sklearn — data splitting & CV ────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)

# ── Sklearn — preprocessing & pipelines ─────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# ── Sklearn — models ─────────────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ── Sklearn — metrics ────────────────────────────────────────────────────────
from sklearn.metrics import (
    recall_score, f1_score, precision_score,
    accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
    classification_report,
)


## Project Constants
All configurable values are defined here. Never hardcode these values inline — change them once here and every cell updates automatically.

In [ ]:
# ── Reproducibility & split configuration ────────────────────────────────────
RANDOM_STATE  = 123
TEST_SIZE     = 0.2

# ── Dataset path (uploaded to Colab session) ─────────────────────────────────
DATA_PATH     = "/content/dataset-uci.csv"

# ── Target column ─────────────────────────────────────────────────────────────
TARGET_COL    = "Gallstone Status"

# ── Feature selection ─────────────────────────────────────────────────────────
K_BEST        = 15          # Top-k features to keep via SelectKBest
CORR_THRESH   = 0.90        # Drop one feature from any pair above this threshold

# ── CV configuration ──────────────────────────────────────────────────────────
N_FOLDS       = 5

# ── Plot output folder (saves figures in Drive if mounted, or /content/) ──────
FIG_DIR       = "/content/figures_gallstone"
os.makedirs(FIG_DIR, exist_ok=True)

print("Constants defined ✅")
print(f"  RANDOM_STATE : {RANDOM_STATE}")
print(f"  TEST_SIZE    : {TEST_SIZE}")
print(f"  K_BEST       : {K_BEST}")
print(f"  CORR_THRESH  : {CORR_THRESH}")
print(f"  N_FOLDS      : {N_FOLDS}")
print(f"  FIG_DIR      : {FIG_DIR}")

Constants defined ✅
  RANDOM_STATE : 123
  TEST_SIZE    : 0.2
  K_BEST       : 15
  CORR_THRESH  : 0.9
  N_FOLDS      : 5
  FIG_DIR      : /content/figures_gallstone


## Environment Verification
Confirm the dataset is readable and matches the expected shape before touching any data.  
**Expected shape:** (319, 38) — 319 patients, 38 features (plus the target column, so 39 total raw columns).

In [ ]:
# ── Verify the CSV is accessible and matches expected dimensions ─────────────
try:
    _df_check = pd.read_csv(DATA_PATH)
    print(f"File loaded successfully ✅")
    print(f"  Shape       : {_df_check.shape}")
    print(f"  Columns     : {_df_check.shape[1]}")
    print(f"  Rows        : {_df_check.shape[0]}")

    # Soft assertion — warns without crashing if shape is unexpected
    expected_rows, expected_cols = 319, 39  # 38 features + 1 target
    if _df_check.shape != (expected_rows, expected_cols):
        print(f"\n⚠️  Unexpected shape. Got {_df_check.shape}, expected ({expected_rows}, {expected_cols}).")
        print("   Check that you uploaded the correct file.")
    else:
        print(f"\nShape matches expected {(expected_rows, expected_cols)} ✅")

    assert TARGET_COL in _df_check.columns, f"Target column '{TARGET_COL}' not found!"
    print(f"Target column '{TARGET_COL}' present ✅")

except FileNotFoundError:
    print(f"❌ File not found at: {DATA_PATH}")
    print("   → Go to Colab sidebar > Files > Upload, then re-run this cell.")

File loaded successfully ✅
  Shape       : (319, 39)
  Columns     : 39
  Rows        : 319

Shape matches expected (319, 39) ✅
Target column 'Gallstone Status' present ✅


## 2. Data Loading & Global Cleaning

### 2.1 Load Data


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}: '{col}'")

Shape: (319, 39)

Column names:
   0: 'Gallstone Status'
   1: 'Age'
   2: 'Gender'
   3: 'Comorbidity'
   4: 'Coronary Artery Disease (CAD)'
   5: 'Hypothyroidism'
   6: 'Hyperlipidemia'
   7: 'Diabetes Mellitus (DM)'
   8: 'Height'
   9: 'Weight'
  10: 'Body Mass Index (BMI)'
  11: 'Total Body Water (TBW)'
  12: 'Extracellular Water (ECW)'
  13: 'Intracellular Water (ICW)'
  14: 'Extracellular Fluid/Total Body Water (ECF/TBW)'
  15: 'Total Body Fat Ratio (TBFR) (%)'
  16: 'Lean Mass (LM) (%)'
  17: 'Body Protein Content (Protein) (%)'
  18: 'Visceral Fat Rating (VFR)'
  19: 'Bone Mass (BM)'
  20: 'Muscle Mass (MM)'
  21: 'Obesity (%)'
  22: 'Total Fat Content (TFC)'
  23: 'Visceral Fat Area (VFA)'
  24: 'Visceral Muscle Area (VMA) (Kg)'
  25: 'Hepatic Fat Accumulation (HFA)'
  26: 'Glucose'
  27: 'Total Cholesterol (TC)'
  28: 'Low Density Lipoprotein (LDL)'
  29: 'High Density Lipoprotein (HDL)'
  30: 'Triglyceride'
  31: 'Aspartat Aminotransferaz (AST)'
  32: 'Alanin Aminotransfera

### 2.2 Quality Check

Inspect data types, missing values, and a statistical summary.  
No transformations yet — only observation and documentation.

In [ ]:
# ── Data types ───────────────────────────────────────────────────────────────
print("=== Data Types ===")
print(df.dtypes)
print()

# ── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
print("=== Missing Values ===")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✅")
print()

# ── Value ranges (numeric summary) ───────────────────────────────────────────
print("=== Statistical Summary ===")
print(df.describe().round(2).to_string())

=== Data Types ===
Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)                   float64
Lean Mass (LM) (%)                                flo

### 2.3 Cleaning Steps

#### 2.3.1 — Column name registry

All full column names are registered here as a single source of truth.  
Future phases reference `COLS['key']` instead of hardcoding long strings.

In [ ]:
# ── Single source of truth for column names ───────────────────────────────────
COLS = {
    # Target
    "target"         : "Gallstone Status",
    # Demographics
    "age"            : "Age",
    "gender"         : "Gender",
    # Comorbidity
    "comorbidity"    : "Comorbidity",
    "cad"            : "Coronary Artery Disease (CAD)",
    "hypothyroidism" : "Hypothyroidism",
    "hyperlipidemia" : "Hyperlipidemia",
    "dm"             : "Diabetes Mellitus (DM)",
    # Anthropometrics
    "height"         : "Height",
    "weight"         : "Weight",
    "bmi"            : "Body Mass Index (BMI)",
    "obesity"        : "Obesity (%)",
    # Body composition
    "tbw"            : "Total Body Water (TBW)",
    "ecw"            : "Extracellular Water (ECW)",
    "icw"            : "Intracellular Water (ICW)",
    "ecf_tbw"        : "Extracellular Fluid/Total Body Water (ECF/TBW)",
    "tbfr"           : "Total Body Fat Ratio (TBFR) (%)",
    "lean_mass"      : "Lean Mass (LM) (%)",
    "protein"        : "Body Protein Content (Protein) (%)",
    "vfr"            : "Visceral Fat Rating (VFR)",
    "bone_mass"      : "Bone Mass (BM)",
    "muscle_mass"    : "Muscle Mass (MM)",
    "tfc"            : "Total Fat Content (TFC)",
    "vfa"            : "Visceral Fat Area (VFA)",
    "vma"            : "Visceral Muscle Area (VMA) (Kg)",
    "hfa"            : "Hepatic Fat Accumulation (HFA)",
    # Lab values
    "glucose"        : "Glucose",
    "cholesterol"    : "Total Cholesterol (TC)",
    "ldl"            : "Low Density Lipoprotein (LDL)",
    "hdl"            : "High Density Lipoprotein (HDL)",
    "triglyceride"   : "Triglyceride",
    "ast"            : "Aspartat Aminotransferaz (AST)",
    "alt"            : "Alanin Aminotransferaz (ALT)",
    "alp"            : "Alkaline Phosphatase (ALP)",
    "creatinine"     : "Creatinine",
    "gfr"            : "Glomerular Filtration Rate (GFR)",
    "crp"            : "C-Reactive Protein (CRP)",
    "hgb"            : "Hemoglobin (HGB)",
    "vitamin_d"      : "Vitamin D",
}

print(f"Column registry defined ✅  ({len(COLS)} entries)")

Column registry defined ✅  (39 entries)


#### 2.3.2 — Verify binary columns

Confirm that `Gender`, `CAD`, `Hypothyroidism`, `Hyperlipidemia`, and `DM` contain only 0/1 values.  
All columns are already numeric — we verify the value range only.

In [ ]:
binary_cols = [
    COLS["gender"],
    COLS["cad"],
    COLS["hypothyroidism"],
    COLS["hyperlipidemia"],
    COLS["dm"],
]

print("=== Binary Column Check ===")
all_ok = True
for col in binary_cols:
    actual  = set(df[col].dropna().unique())
    allowed = {0, 1}
    status  = "✅" if actual <= allowed else "⚠️ "
    if actual > allowed:
        all_ok = False
    print(f"  {status} {col}: {sorted(actual)}")

print()
if all_ok:
    print("All binary columns confirmed as 0/1 ✅ — no encoding needed.")
else:
    print("⚠️  Some columns have unexpected values — inspect above.")

=== Binary Column Check ===
  ✅ Gender: [np.int64(0), np.int64(1)]
  ✅ Coronary Artery Disease (CAD): [np.int64(0), np.int64(1)]
  ✅ Hypothyroidism: [np.int64(0), np.int64(1)]
  ✅ Hyperlipidemia: [np.int64(0), np.int64(1)]
  ✅ Diabetes Mellitus (DM): [np.int64(0), np.int64(1)]

All binary columns confirmed as 0/1 ✅ — no encoding needed.


#### 2.3.3 — Check `Comorbidity` redundancy

**Hypothesis:** `Comorbidity` is a derived column from `CAD`, `Hypothyroidism`, `Hyperlipidemia`, and `DM`.  
The statistical summary showed `max = 3`, suggesting it may be a **count** of active comorbidities rather than a simple OR flag.  
We test both hypotheses. If either is confirmed, `Comorbidity` is redundant and will be dropped in Phase 4.

In [ ]:
components = [
    COLS["cad"],
    COLS["hypothyroidism"],
    COLS["hyperlipidemia"],
    COLS["dm"],
]

comorbidity_col = COLS["comorbidity"]

# ── Hypothesis A: Comorbidity = OR (binary flag) ──────────────────────────────
derived_or    = (df[components].sum(axis=1) > 0).astype(int)
match_or      = (derived_or == df[comorbidity_col]).sum()

# ── Hypothesis B: Comorbidity = SUM (count) ───────────────────────────────────
derived_sum   = df[components].sum(axis=1).astype(int)
match_sum     = (derived_sum == df[comorbidity_col]).sum()

total = len(df)

print(f"=== Comorbidity Redundancy Check ===")
print(f"  Unique values in Comorbidity : {sorted(df[comorbidity_col].unique())}")
print()
print(f"  Hypothesis A — binary OR flag:")
print(f"    Matching rows : {match_or} / {total}  ({match_or/total:.1%})")
print()
print(f"  Hypothesis B — sum / count of components:")
print(f"    Matching rows : {match_sum} / {total}  ({match_sum/total:.1%})")
print()

if match_sum == total:
    print("✅ CONFIRMED: Comorbidity = SUM of components (count).")
    print("   → Redundant. Will be DROPPED in Phase 4.")
    COMORBIDITY_IS_DERIVED = True
elif match_or == total:
    print("✅ CONFIRMED: Comorbidity = OR of components (binary flag).")
    print("   → Redundant. Will be DROPPED in Phase 4.")
    COMORBIDITY_IS_DERIVED = True
elif max(match_or, match_sum) / total >= 0.95:
    print("⚠️  Near-perfect match — likely derived with minor data entry errors.")
    print("   → Will be DROPPED in Phase 4.")
    COMORBIDITY_IS_DERIVED = True
else:
    print("❌ Comorbidity does NOT appear to be derived from its components.")
    print("   → Retains independent information. Keep for now.")
    COMORBIDITY_IS_DERIVED = False

print(f"\nCOMORBIDITY_IS_DERIVED = {COMORBIDITY_IS_DERIVED}")

=== Comorbidity Redundancy Check ===
  Unique values in Comorbidity : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

  Hypothesis A — binary OR flag:
    Matching rows : 282 / 319  (88.4%)

  Hypothesis B — sum / count of components:
    Matching rows : 278 / 319  (87.1%)

❌ Comorbidity does NOT appear to be derived from its components.
   → Retains independent information. Keep for now.

COMORBIDITY_IS_DERIVED = False


#### 2.3.4 — Verify target column

Confirm `Gallstone Status` contains only 0 and 1, and record the class distribution.  
Class balance here determines whether `stratify=y` is critical in the split.

In [ ]:
print("=== Target Column Verification ===")
print(f"  Column        : '{COLS['target']}'")
print(f"  Unique values : {sorted(df[COLS['target']].unique())}")
print()

counts = df[COLS["target"]].value_counts().sort_index()
props  = df[COLS["target"]].value_counts(normalize=True).sort_index()

for cls in counts.index:
    print(f"  Class {cls} : {counts[cls]} rows  ({props[cls]:.1%})")

print()

# Validity check
if set(df[COLS["target"]].unique()) <= {0, 1}:
    print("✅ Target is valid binary (0 / 1).")
else:
    print("⚠️  Unexpected values in target — inspect manually.")

# Balance check
majority = props.max()
if majority > 0.65:
    print(f"⚠️  Class imbalance detected — majority class is {majority:.1%}.")
    print("   → stratify=y in the split is critical.")
else:
    print(f"✅ Classes are approximately balanced ({majority:.1%} majority).")
    print("   → stratify=y still used as best practice for small datasets.")

=== Target Column Verification ===
  Column        : 'Gallstone Status'
  Unique values : [np.int64(0), np.int64(1)]

  Class 0 : 161 rows  (50.5%)
  Class 1 : 158 rows  (49.5%)

✅ Target is valid binary (0 / 1).
✅ Classes are approximately balanced (50.5% majority).
   → stratify=y still used as best practice for small datasets.
